# ImageNet validation

Loads the compressed ~4 MiB checkpoint and reports ImageNet validation top-1 and top-5 accuracy.

Expected repository files: `main.py`, `model.py`, and `linearggm.py`.


In [1]:
import os
from contextlib import nullcontext

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

from main import Config
from model import VisionTransformer
from linearggm import LinearGGM


CHECKPOINT_PATH = "runs/best_seed_G_frozen_Wb_1bit_fp16.pth"
IMAGENET_DIR = "../../imagenet_download"
BATCH_SIZE = 256
NUM_WORKERS = 8
USE_AMP = torch.cuda.is_available()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"Device: {device}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Checkpoint size: {os.path.getsize(CHECKPOINT_PATH) / 1024**2:.2f} MiB")


Device: cuda
Checkpoint: runs/best_seed_G_frozen_Wb_1bit_fp16.pth
Checkpoint size: 4.27 MiB


In [2]:
class FrozenLinearGGM(nn.Module):
    def __init__(
        self,
        *,
        in_features,
        out_features,
        N,
        centralize_x,
        G_seed,
        G,
        W_b,
        gain,
        bias=None,
    ):
        super().__init__()
        self.in_features = int(in_features)
        self.out_features = int(out_features)
        self.N = int(N)
        self.centralize_x = bool(centralize_x)
        self.G_seed = int(G_seed)

        self.register_buffer("W_b", W_b.to(torch.int8).contiguous())
        self.register_buffer("gain", gain.to(torch.float32).contiguous())
        self.register_buffer("G", G.to(torch.float32).contiguous(), persistent=False)

        if bias is None:
            self.bias = None
        else:
            self.register_buffer("bias", bias.to(torch.float32).contiguous())

    def forward(self, x):
        x_eff = x - x.mean(dim=-1, keepdim=True) if self.centralize_x else x.float()
        x_b = (x_eff @ self.G.transpose(-1, -2)).sign()
        y = (x_b @ self.W_b.to(x_b.dtype)) / self.N
        y = y * self.gain
        return y if self.bias is None else y + self.bias


def load_checkpoint_cpu(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")


def replace_submodule(root, module_name, replacement):
    if "." in module_name:
        parent_name, child_name = module_name.rsplit(".", 1)
        parent = root.get_submodule(parent_name)
    else:
        parent = root
        child_name = module_name
    setattr(parent, child_name, replacement)


def unpack_binary_tensor(packed, metadata):
    packed = packed.detach().cpu().to(torch.uint8).flatten()
    shifts = torch.arange(8, dtype=torch.int16)
    bits = (((packed.to(torch.int16)[:, None] >> shifts[None, :]) & 1).flatten())
    bits = bits[: metadata["numel"]]
    return (bits.to(torch.int8) * 2 - 1).reshape(metadata["shape"]).contiguous()


def build_frozen_architecture(artifact):
    model = VisionTransformer(Config(**artifact["config"]))
    use_exact_g = artifact["g_storage"] == "exact_deduplicated"

    for module_name, spec in artifact["linear_ggm_specs"].items():
        original_layer = model.get_submodule(module_name)
        if not isinstance(original_layer, LinearGGM):
            raise TypeError(f"{module_name} is not a LinearGGM in the base model")

        if use_exact_g:
            G = artifact["g_bank"][artifact["g_map"][module_name]].cpu().contiguous()
        else:
            if int(original_layer.G_seed) != int(spec["G_seed"]):
                raise RuntimeError(f"G seed mismatch for {module_name}")
            G = original_layer.G.detach().cpu().contiguous()

        replacement = FrozenLinearGGM(
            in_features=spec["in_features"],
            out_features=spec["out_features"],
            N=spec["N"],
            centralize_x=spec["centralize_x"],
            G_seed=spec["G_seed"],
            G=G,
            W_b=torch.zeros(spec["N"], spec["out_features"], dtype=torch.int8),
            gain=torch.ones(spec["out_features"], dtype=torch.float32),
            bias=(
                torch.zeros(spec["out_features"], dtype=torch.float32)
                if spec["has_bias"]
                else None
            ),
        )
        replace_submodule(model, module_name, replacement)

    return model


def load_compressed_model(path, target_device):
    artifact = load_checkpoint_cpu(path)
    model = build_frozen_architecture(artifact)

    if artifact["wb_storage"] != "packed_1bit":
        raise ValueError(f"Expected packed_1bit checkpoint, got {artifact['wb_storage']}")

    result = model.load_state_dict(artifact["state_dict"], strict=False)
    expected_missing = {
        f"{module_name}.W_b" for module_name in artifact["linear_ggm_specs"]
    }

    if set(result.missing_keys) != expected_missing or result.unexpected_keys:
        raise RuntimeError(
            f"Checkpoint loading failed. Missing: {result.missing_keys}; "
            f"unexpected: {result.unexpected_keys}"
        )

    with torch.no_grad():
        for module_name in artifact["linear_ggm_specs"]:
            layer = model.get_submodule(module_name)
            layer.W_b.copy_(
                unpack_binary_tensor(
                    artifact["packed_wb"][module_name],
                    artifact["packed_wb_metadata"][module_name],
                )
            )

    return model.to(target_device).eval(), Config(**artifact["config"])


In [3]:
model, config = load_compressed_model(CHECKPOINT_PATH, device)

val_dir = os.path.join(IMAGENET_DIR, "val")
if not os.path.isdir(val_dir):
    raise FileNotFoundError(
        f"ImageNet validation directory not found: {val_dir}\n"
        "Expected layout: imagenet_download/val/<class>/<image>"
    )

val_transform = transforms.Compose([
    transforms.Resize(
        int(config.image_size * 256 / 224),
        interpolation=transforms.InterpolationMode.BICUBIC,
    ),
    transforms.CenterCrop(config.image_size),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
    drop_last=False,
)

print(f"Validation images: {len(val_dataset):,}")
print(f"Classes: {len(val_dataset.classes):,}")


Validation images: 50,000
Classes: 1,000


In [4]:
@torch.inference_mode()
def evaluate(model, loader):
    top1_correct = 0
    top5_correct = 0
    total = 0

    progress = tqdm(loader, desc="ImageNet validation")
    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        amp_context = (
            torch.autocast(device_type="cuda", dtype=torch.float16)
            if USE_AMP and device.type == "cuda"
            else nullcontext()
        )

        with amp_context:
            logits = model(images)

        top5 = logits.topk(k=5, dim=1).indices
        top1_correct += (top5[:, 0] == labels).sum().item()
        top5_correct += top5.eq(labels[:, None]).any(dim=1).sum().item()
        total += labels.size(0)

        progress.set_postfix(
            top1=f"{100 * top1_correct / total:.3f}%",
            top5=f"{100 * top5_correct / total:.3f}%",
        )

    return 100 * top1_correct / total, 100 * top5_correct / total


top1, top5 = evaluate(model, val_loader)

print("\nResults")
print(f"Top-1 accuracy: {top1:.4f}%")
print(f"Top-5 accuracy: {top5:.4f}%")


ImageNet validation:   0%|          | 0/196 [00:00<?, ?it/s]


Results
Top-1 accuracy: 56.1880%
Top-5 accuracy: 79.6120%
